In [1]:
import os, pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# 读数据
save_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")
df = pd.read_csv(f"{save_dir}/features_labeled.csv", parse_dates=["Date"])

feature_cols = ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d',
                'rsi_dist', 'high_20d_ratio', 'vol_ratio']

# 按日期组织成列表：每个元素是 (特征矩阵[5,8], rank向量[5])
groups = []
for date, group in df.groupby("Date"):
    group = group.sort_values("ticker").reset_index(drop=True)
    feats = torch.tensor(group[feature_cols].values, dtype=torch.float32)  # [5, 8]
    ranks = torch.tensor(group["rank"].values, dtype=torch.float32)        # [5]
    groups.append((feats, ranks))

print(f"总共 {len(groups)} 个交易日")
print(f"每日特征 shape: {groups[0][0].shape}")
print(f"某日 rank: {groups[0][1]}")

总共 40 个交易日
每日特征 shape: torch.Size([5, 8])
某日 rank: tensor([5., 2., 3., 1., 4.])


In [2]:
# 复用同一个 scorer 结构
class ListNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        # x: [n, 8]，返回 [n] 分数
        return self.scorer(x).squeeze(1)

def listnet_loss(scores, ranks):
    # scores: [n]  模型输出的原始分数
    # ranks:  [n]  真实排名（1=最好，5=最差）
    # 转成"越大越好"的得分：用负rank，这样rank=1对应最高分
    rank_scores = -ranks
    p_true  = torch.softmax(rank_scores, dim=0)  # 真实分布
    p_pred  = torch.softmax(scores, dim=0)        # 预测分布
    # cross-entropy: -sum(p_true * log(p_pred))
    loss = -torch.sum(p_true * torch.log(p_pred + 1e-9))
    return loss

# 标准化（用 Week14 的 scaler fit 过的，这里重新 fit 原始特征）
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
all_feats = df[feature_cols].values
scaler.fit(all_feats)

groups_scaled = []
for feats, ranks in groups:
    feats_scaled = torch.tensor(
        scaler.transform(feats.numpy()), dtype=torch.float32)
    groups_scaled.append((feats_scaled, ranks))

# 切分：前32天训练，后8天验证
train_groups = groups_scaled[:32]
val_groups   = groups_scaled[32:]
print(f"train: {len(train_groups)} 天，val: {len(val_groups)} 天")

# 训练
model_ln = ListNet(input_dim=8)
optimizer = torch.optim.Adam(model_ln.parameters(), lr=1e-3)

for epoch in range(100):
    model_ln.train()
    total_loss = 0
    for feats, ranks in train_groups:
        optimizer.zero_grad()
        scores = model_ln(feats)
        loss = listnet_loss(scores, ranks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        model_ln.eval()
        with torch.no_grad():
            val_loss = sum(listnet_loss(model_ln(f), r).item()
                          for f, r in val_groups) / len(val_groups)
        print(f"Epoch {epoch+1:3d} | train_loss: {total_loss/len(train_groups):.4f} | val_loss: {val_loss:.4f}")

train: 32 天，val: 8 天
Epoch  20 | train_loss: 1.5207 | val_loss: 1.5782
Epoch  40 | train_loss: 1.4092 | val_loss: 1.6474
Epoch  60 | train_loss: 1.3689 | val_loss: 1.6843
Epoch  80 | train_loss: 1.2872 | val_loss: 1.7407
Epoch 100 | train_loss: 1.2875 | val_loss: 1.7792


In [3]:
model_ln = ListNet(input_dim=8)
optimizer = torch.optim.Adam(model_ln.parameters(), lr=1e-3)

best_val_loss = float("inf")
best_epoch = 0
patience = 20
no_improve = 0

for epoch in range(200):
    model_ln.train()
    # --- batch：整批天数一起更新 ---
    optimizer.zero_grad()
    total_loss = sum(listnet_loss(model_ln(f), r) for f, r in train_groups)
    total_loss.backward()
    optimizer.step()
    
    model_ln.eval()
    with torch.no_grad():
        val_loss = sum(listnet_loss(model_ln(f), r).item()
                       for f, r in val_groups) / len(val_groups)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = {k: v.clone() for k, v in model_ln.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    
    if (epoch + 1) % 20 == 0:
        train_avg = total_loss.item() / len(train_groups)
        print(f"Epoch {epoch+1:3d} | train_loss: {train_avg:.4f} | val_loss: {val_loss:.4f} | best: {best_val_loss:.4f} @ ep{best_epoch}")
    
    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}，best val_loss: {best_val_loss:.4f} @ ep{best_epoch}")
        break

model_ln.load_state_dict(best_state)
print("已恢复最优权重")

Epoch  20 | train_loss: 1.5785 | val_loss: 1.6132 | best: 1.6132 @ ep20
Epoch  40 | train_loss: 1.5512 | val_loss: 1.6062 | best: 1.6062 @ ep40
Epoch  60 | train_loss: 1.5421 | val_loss: 1.6003 | best: 1.6003 @ ep60
Epoch  80 | train_loss: 1.4933 | val_loss: 1.5966 | best: 1.5966 @ ep80
Epoch 100 | train_loss: 1.4700 | val_loss: 1.5964 | best: 1.5964 @ ep100
Epoch 120 | train_loss: 1.4776 | val_loss: 1.6046 | best: 1.5964 @ ep100

Early stopping at epoch 120，best val_loss: 1.5964 @ ep100
已恢复最优权重


In [5]:
class RankNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, fi, fj):
        si = self.scorer(fi)  # [B, 1]
        sj = self.scorer(fj)  # [B, 1]
        return torch.sigmoid(si - sj).squeeze(1)  # [B]
    
    def score(self, f):
        # 推理时单独给每只股票打分
        return self.scorer(f).squeeze(1)


In [6]:
def ndcg_at_k(pred_scores, true_ranks, k=3):
    # true_ranks: 1=最好，数字越小越好
    # 按预测分数降序排列
    order = torch.argsort(pred_scores, descending=True)
    true_relevance = (6 - true_ranks).float()  # 转成越大越好：rank1→5, rank5→1
    
    dcg = sum((true_relevance[order[i]] / np.log2(i + 2)).item()
              for i in range(k))
    
    ideal_order = torch.argsort(true_relevance, descending=True)
    idcg = sum((true_relevance[ideal_order[i]] / np.log2(i + 2)).item()
               for i in range(k))
    
    return dcg / idcg if idcg > 0 else 0.0

# 加载 RankNet
ckpt = torch.load(f"{save_dir}/ranknet_v1.pth")
model_rn = RankNet(input_dim=8)
model_rn.load_state_dict(ckpt["model_state"])
model_rn.eval()

# 在 val_groups 上对比
rn_scores, ln_scores = [], []
for feats, ranks in val_groups:
    with torch.no_grad():
        s_rn = model_rn.score(feats)
        s_ln = model_ln(feats)
    rn_scores.append(ndcg_at_k(s_rn, ranks, k=3))
    ln_scores.append(ndcg_at_k(s_ln, ranks, k=3))

print(f"{'模型':<12} {'NDCG@3 mean':>12} {'NDCG@3 std':>12}")
print("-" * 38)
print(f"{'RankNet':<12} {np.mean(rn_scores):>12.4f} {np.std(rn_scores):>12.4f}")
print(f"{'ListNet':<12} {np.mean(ln_scores):>12.4f} {np.std(ln_scores):>12.4f}")
print(f"\n随机基准 NDCG@3 ≈ {1/3:.4f}（随机排序期望值）")

模型            NDCG@3 mean   NDCG@3 std
--------------------------------------
RankNet            0.9114       0.0904
ListNet            0.7103       0.1375

随机基准 NDCG@3 ≈ 0.3333（随机排序期望值）


In [7]:
# 保存 ListNet
torch.save({
    "model_state": model_ln.state_dict(),
    "hparams": {"input_dim": 8, "feature_cols": feature_cols}
}, f"{save_dir}/listnet_v1.pth")
print("已保存 listnet_v1.pth")

# 直觉验证：取 val 第一天，看预测排序 vs 真实排序
feats, ranks = val_groups[0]
tickers_that_day = (df.groupby("Date")
                      .apply(lambda g: g.sort_values("ticker")["ticker"].tolist())
                      .iloc[32])

with torch.no_grad():
    scores = model_ln(feats)

result = pd.DataFrame({
    "ticker":     tickers_that_day,
    "pred_score": scores.numpy().round(4),
    "pred_rank":  pd.Series(scores.numpy()).rank(ascending=False).astype(int).tolist(),
    "true_rank":  ranks.numpy().astype(int).tolist()
})
result = result.sort_values("pred_rank")
print("\nval 第1天排序结果：")
print(result.to_string(index=False))

已保存 listnet_v1.pth

val 第1天排序结果：
ticker  pred_score  pred_rank  true_rank
  NVDA      0.1285          1          3
 GOOGL     -0.0758          2          1
  AMZN     -0.3427          3          2
  MSFT     -0.4587          4          5
  AAPL     -0.8129          5          4


/var/folders/lv/mnjxv0d53_xczbffr49m5z_40000gn/T/ipykernel_62946/2655979058.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("ticker")["ticker"].tolist())
